# BayMorph — Fit an Observed Single-Band Image

This notebook executes the reviewed analysis defined by
`02A_Observed_Assessment_and_Config.ipynb`.

The YAML is the reproducible contract between assessment and inference.
This notebook is deliberately fit-only: it validates and consumes the
accepted preparation and PSF products without rebuilding either one.
The same YAML can also be consumed by a terminal Python session.


In [ ]:
import json
from pathlib import Path

import yaml

from baymorph import BayMorphRunner


## 1. Run the fit-only workflow

Select the YAML reviewed and written by notebook 02A. Relative paths are
resolved from the BayMorph project root. The cell prints the resolved
sampler and model set before starting the expensive work. Preparation
and PSF construction are not called from this notebook.

For Nautilus, `filepath: auto` resolves to a signed checkpoint inside
each model directory. Re-running an incomplete compatible fit resumes
that checkpoint; a changed scientific target is rejected. Completed
products are protected by `outputs.overwrite`.


In [ ]:
config_path = Path("configs/observed_galaxy.yaml")

runner = BayMorphRunner(config_path)
sampler_config = runner.config["fit"]["sampler"]
models = runner.config["fit"]["models"]

print("Resolved sampler configuration:")
print(
    yaml.safe_dump(
        sampler_config,
        sort_keys=False,
    )
)
print("Models:", models)
if sampler_config["name"] == "emcee":
    print(
        "emcee is posterior-only; this run will not produce logZ."
    )

fit_results = runner.fit()
runner.diagnostics()

results_by_model = (
    fit_results
    if isinstance(fit_results, dict)
    else {fit_results.model_name: fit_results}
)


### Terminal/Python equivalent

The equivalent non-notebook workflow uses the same fit-only calls:

    runner = BayMorphRunner(config_path)
    fit_results = runner.fit()
    runner.diagnostics()

Do not run both interfaces concurrently against the same checkpoint.


## 2. Inspect the result

Every backend supplies posterior samples, provenance, goodness-of-fit
statistics, best-fitting parameters, prior-boundary diagnostics, and
persisted products. Evidence is displayed only when the selected backend
estimates it.

For emcee, inspect trace stability, autocorrelation estimates, MCMC
effective sample size, acceptance fractions, agreement across independent
seeds, residuals, and prior-boundary diagnostics. Independent seeds require
distinct analysis names and fresh checkpoints. A checkpoint is a
continuation mechanism, not evidence of convergence. Conservative
compatibility checks reject resume after BayMorph source, environment,
data, PSF, likelihood, prior, model, or sampler changes.


In [ ]:
fit_summaries = {}

for model_name, fit in results_by_model.items():
    inference = fit.posterior
    fit_summaries[model_name] = fit.summary()
    print(f"\n{model_name}")

    if inference.evidence is None:
        print("Evidence: unavailable (posterior-only backend)")
        print(
            "Effective sample size:",
            inference.effective_sample_size,
        )
    else:
        print(f"logZ: {inference.logz}")
        print(f"logZ uncertainty: {inference.logz_error}")

comparison_path = (
    runner.run_dir / "model_comparison" / "bayes_factor.json"
)
comparison = (
    json.loads(comparison_path.read_text())
    if comparison_path.is_file()
    else None
)

{"fits": fit_summaries, "comparison": comparison}
